# Проект "Интернет-магазин"

## Описание проекта

В этом проекте будут использованы данные интернет-магазина BitMotion Kit, в котором продаются геймифицированные товары для тех, кто ведёт здоровый образ жизни. У него есть своя целевая аудитория, даже появились хиты продаж: эспандер со счётчиком и напоминанием, так и подстольный велотренажёр с Bluetooth.

В будущем компания хочет расширить ассортимент товаров. Но перед этим нужно решить одну проблему. Интерфейс онлайн-магазина слишком сложен для пользователей — об этом говорят отзывы.

Чтобы привлечь новых клиентов и увеличить число продаж, владельцы магазина разработали новую версию сайта и протестировали его на части пользователей. По задумке, это решение доказуемо повысит количество пользователей, которые совершат покупку и увеличит конверсию в покупку как минимум на три процентных пункта.

## Цели и задачи проекта

Цель исследования - оценить улучшение конверсии в покупку после изменения интерфейса онлайн-магазина и проверить результаты проведенного А/В теста.

Задачи, поставленные в проекте:
- Оценить корректность проведения теста
- Провести оценку результатов теста

## План работы

- Загрузка данных
- Оценка корректности проведения теста
- Определение горизонта анализа
- Оценка достаточности выборки и пользовательской активности¶
- Оценка результатов A/B-тестирования
- Итоги теста

## Загрузка данных

In [1]:
# Импортируем библиотеки для анализа данных
import pandas as pd
import numpy as np

# Загружаем библиотеку для визуализации данных
import matplotlib.pyplot as plt

# Импортируем библиотеку для проверки гипотез
from scipy import stats
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize
from statsmodels.stats.proportion import proportions_ztest

In [2]:
# Загружаем данные
participants = pd.read_csv('https://code.s3.yandex.net/datasets/ab_test_participants.csv')
events = pd.read_csv('https://code.s3.yandex.net/datasets/ab_test_events.zip',
                     parse_dates=['event_dt'], low_memory=False)

## Оценка корректности проведения теста

Выделим пользователей, участвующих в тесте, и проверим:

   - соответствие требованиям технического задания,

   - равномерность распределения пользователей по группам теста,

   - отсутствие пересечений между группами

   - отсутствие пересечений с конкурирующим тестом (нет пользователей, участвующих одновременно в двух тестовых группах).

In [3]:
# Параметры
START_DATE = '2020-12-07'          # дата запуска теста
STOP_DATE = '2021-01-04'           # дата остановки набора
EXPECTED_USERS = 6000              # ожидаемое число участников
MDE = 0.03                         # ожидаемый эффект, абсолютный (+3 п.п.)
BASE_CR = 0.30                     # базовый показатель конверсии
ALPHA = 0.05                       # уровень значимости
POWER = 0.80                       # мощность теста
HORIZON_DAYS = 7                   # горизонт дней после регистрации

In [4]:
# Количество участников теста
part = participants[participants['ab_test'] == 'interface_eu_test'].copy()
print('Участников теста:', part['user_id'].nunique())

Участников теста: 10850


In [5]:
# Равномерность групп
print(part.groupby('group')['user_id'].nunique())

cnt = part.groupby('group')['user_id'].nunique()
print('Доли групп:\n', cnt / cnt.sum())

group
A    5383
B    5467
Name: user_id, dtype: int64
Доли групп:
 group
A    0.496129
B    0.503871
Name: user_id, dtype: float64


In [6]:
# Пересечение пользователей групп А и В одного теста
multi = part.groupby('user_id')['group'].nunique()
print('В обеих группах A и B:', (multi > 1).sum())

В обеих группах A и B: 0


In [7]:
# Пересечение с конкурирующим тестом
other_ids = set(participants.loc[participants['ab_test'] == 'recommender_system_test', 'user_id'])
both_tests = set(part['user_id']) & other_ids
print('Пересечение с конкурирующим тестом:', len(both_tests))

Пересечение с конкурирующим тестом: 887


Есть пересечения между тестами, таких пользователей следует исключить из теста.

## Определение горизонта анализа

Оставим только события, связанные с участвующими в изучаемом тесте пользователями

In [8]:
# Оставляем только тех пользователей, которых нет в конкурирующем тесте
part = part[~part['user_id'].isin(other_ids)]

# Оставляем события, связанные с участвующими в изучаемом тесте пользователями
ev = events[events['user_id'].isin(part['user_id'])].copy() 

Определим горизонт анализа: рассчитаем время (lifetime) совершения события пользователем после регистрации и оставим только те события, которые были выполнены в течение первых семи дней с момента регистрации

In [9]:
# Находим дату регистрации каждого пользователя
reg = (ev[ev['event_name'] == 'registration']
       .groupby('user_id')['event_dt'].min().rename('reg_dt'))

# Присоединяем дату регистрации к датафрейму с событиями
ev = ev.merge(reg, on='user_id', how='inner')

# Находим lifetime каждого события в днях
ev['lifetime'] = (ev['event_dt'] - ev['reg_dt']).dt.total_seconds() / 86400

# Фильтр по горизонту анализа
ev = ev[(ev['lifetime'] >= 0) & (ev['lifetime'] < HORIZON_DAYS)]

# Добавляем к событиям группу пользователя
ev = ev.merge(part[['user_id', 'group']].drop_duplicates('user_id'), on='user_id')

print('Событий в горизонте:', len(ev))
print('Минимальная дата события:', ev['event_dt'].min())
print('Максимальная дата события:', ev['event_dt'].max())

Событий в горизонте: 63805
Минимальная дата события: 2020-12-06 14:10:01
Максимальная дата события: 2020-12-29 23:41:16


## Оценка достаточности выборки и пользовательской активности

Оценим достаточность выборки для получения статистически значимых результатов A/B-теста.

In [10]:
# Считаем размер выборки
effect = proportion_effectsize(BASE_CR + MDE, BASE_CR)
n_needed = NormalIndPower().solve_power(effect, power=POWER, alpha=ALPHA,
                                        ratio=1, alternative='two-sided')

print(f'Нужно на группу: {int(np.ceil(n_needed))}')

Нужно на группу: 3762


Пользователей в выборке достаточно для получения статистически значимых результатов A/B-теста

Рассчитаем для каждой группы количество посетителей, сделавших покупку, общее количество посетителей и конверсию.

In [11]:
# Все зарегистрированные пользователи
all_users = part.drop_duplicates('user_id').groupby('group')['user_id'].nunique()

# Пользователи, у которых были события
active = ev.groupby('group')['user_id'].nunique()

# Пользователи, совершившие покупку
buyers = ev[ev['event_name'] == 'purchase'].groupby('group')['user_id'].nunique()

# Сводная таблица
res = pd.DataFrame({'registered': all_users, 'with_events': active, 'buyers': buyers}).fillna(0)

# Считаем конверсию
res['cr_reg'] = res['buyers'] / res['registered']

print(res)

       registered  with_events  buyers    cr_reg
group                                           
A            4952         4952    1377  0.278069
B            5011         5011    1480  0.295350


In [12]:
# Количество пользователей по событиям и группам
funnel = (ev.groupby(['group', 'event_name'])['user_id'].nunique().unstack(0))

# Считаем конверсию на каждом этапе
funnel = funnel.div(res['registered'], axis=1)

print(funnel.sort_values('A', ascending=False))

group                A         B
event_name                      
login         1.000000  0.999800
registration  1.000000  1.000000
product_page  0.627423  0.635402
product_cart  0.281704  0.264817
purchase      0.278069  0.295350


Мы можем заметить улучшение конверсии в покупку в тестовой группе, однако, чтобы подтвердить статистическую значимость этого изменения, необходимо провести статистический тест.

## Оценка результатов A/B-тестирования

**Целевая метрика:** 
- Конверсия в покупку.

**Гипотеза H0**: 
- Новый интерфейс сайта не влияет на уровень конверсии в покупку (конверсия в группе В равна конверсии в группе А: CR_В = CR_А).

**Гипотеза H1**: 
- Новый интерфейс сайта увеличивает конверсию в покупку как минимум на 3 п.п. (конверсия в группе В статистически значимо выше, чем в группе A: CR_В > CR_А).

Для метрики конверсии подходящим статистическим тестом будет Z-тест пропорций.

In [13]:
# Число успехов (покупок)
succ = np.array([res.loc['B', 'buyers'], res.loc['A', 'buyers']])

# Число наблюдений (регистраций)
nobs = np.array([res.loc['B', 'registered'], res.loc['A', 'registered']])

# Z-тест пропорций
z, p = proportions_ztest(succ, nobs, alternative='larger')

print(f'Изменение конверсии={(succ[0]/nobs[0] - succ[1]/nobs[1]):.3f}')
print(f'p-value={p:.4f}')
if p < ALPHA:
    print(f'Различие конверсии между группами статистически значимо.'
          f'Отвергаем нулевую гипотезу о равенстве долей.')
else:
    print(f'Статистически значимого различия конверсии между группами не обнаружено.'
          f'Нет оснований отвергать нулевую гипотезу о равенстве долей.')

Изменение конверсии=0.017
p-value=0.0283
Различие конверсии между группами статистически значимо.Отвергаем нулевую гипотезу о равенстве долей.


## Итоги теста

- Статистическим тестом был выбран Z-тест пропорций
- Размер выборки оказался достаточным
- Тест проведен корректно
- Статистическая значимость изменения конверсии подтвердилась тестом: новая версия сайта улучшила конверсию в покупку в тестовой группе
- Ожидаемый эффект увеличения конверсии (+3 п.п.) достигнут не был. Реальный эффект составил +1.7 п.п.
- Ситуация говорит о том, что, хоть эффект изменения статистически значим, он не достигает порога практической значимости (MDE): требуется оценка бизнес‑ценности выявленного эффекта